In [ ]:
"""
# Earlier Variables and Explanations

model: Pre-trained DARNet model

The Variables the model trained by: 
Xtr: shape:(11460,64,750) | PreApplied BatchNorm(dim=1,2)
ytr: shape:(11460,) unique:[0,1]

The Variables haven't seen by the model.
Xte: shape:(764,64,750) | PreApplied BatchNorm(dim=1,2)
yte: shape:(764,) unique:[0,1]

ytr_: shape:(11460,) mean:0.0947 std:1.9512
yte_: shape:(764,)   mean:0.0884 std:2.0590

ytr_ and yte_ are the outputs of pre-trained model that Xtr and Xte are given as inputs respectively.
the model returns (batch, 2) sized outputs and we used y_[batch_idx] = out[:,1] - out[:,0] to esteblish the variables ytr_ and yte_

pre-trained model score:
>>> ((yte_>0)==yte).float().mean()
tensor(0.7618)  # ChanceLevel:50%
"""

((yte_>0)==yte).float().mean()

tensor(0.7618)

In [89]:
class CUTHDMR:
    def __init__(self, model, Xtr):
        self.Xtr = Xtr  # tensor:(11460,64,750)
        self.vay = self.Xtr.mean(dim=0, keepdim=True)   # tensor:(1,64,750)
        self._model = model

        self.f0 = self.model(self.vay)
        
    def model(self,X):
        self._model.eval()
        with torch.no_grad():
            logits = self._model(X.to(utils.device))
            logits = logits.log_softmax(dim=1)
            logit_pos = logits[:, 1] - logits[:, 0]
            out = logit_pos.detach().cpu()
            # out = self._model(X.to(utils.device)).softmax(dim=1)[:,1].detach().cpu()
        return out

    def _calculate_fi(self, x, batch_size=128): 
        # x: shape:(64,750)

        vayxes = self.vay.repeat(64*750, 1, 1) # tensor:48000,64,750
        for i in range(64*750):
            vayxes.reshape(-1,64*750)[i,i] = x.reshape(-1)[i]
        
        outputs = torch.zeros(64*750, device=self.Xtr.device)
        self.vayxes = vayxes

        # Set your batch size to 128 (or higher, e.g., 256/512 if your GPU has headroom)
        batch_size = 128
        dataset = TensorDataset(vayxes)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        for i,batch in enumerate(loader):
            out = self.model(batch[0])
            outputs[i*batch_size:i*batch_size+len(out)] = out
            if (i+1)%100==0 or i+1==len(loader):
                print(f"{i+1}/{len(loader)}")
                
        self.fi = outputs      # (48000,)
        self.fi -= chdmr.f0

    def run(self, x):
        self._calculate_fi(x)
        return self.fi.sum() + self.f0

chdmr = CUTHDMR(model, Xtr)     #### initilized for only y=avarage ####

for i in [1,3,4,5,12,701]:       
    mimic_yte = chdmr.run(Xte[0])  #### each run took 40s seconds ####

    print(f"test_index:{i:3d} Mimic_out:{mimic_yte.item():9.4f} Model_out:{yte_[i]:8.4f}, Real_out:{yte[i]}")

# Note

data initilized...
100/375
200/375
300/375
375/375
test_index:  1 Mimic_out:   4.9454 Model_out: -3.0936, Real_out:1.0
data initilized...
100/375
200/375
300/375
375/375
test_index:  3 Mimic_out:   4.9454 Model_out: -1.3469, Real_out:1.0
data initilized...
100/375
200/375
300/375
375/375
test_index:  4 Mimic_out:   4.9454 Model_out: -1.9063, Real_out:1.0
data initilized...
100/375
200/375
300/375
375/375
test_index:  5 Mimic_out:   4.9454 Model_out:  0.4298, Real_out:1.0
data initilized...
100/375
200/375
300/375
375/375
test_index: 12 Mimic_out:   4.9454 Model_out: -0.7022, Real_out:1.0
data initilized...
100/375
200/375
300/375
375/375
test_index:701 Mimic_out:   4.9454 Model_out:  0.5151, Real_out:1.0


In [ ]:
class ANOVAHDMR:
    def __init__(self, Xtr, ytr):
        self.Xtr = Xtr              # tensor:(11460,64,750)
        self.ytr = ytr              # tensor:(11460,)
        self.f0 = self.ytr.mean().item()
        
    
    def _calculate_fi(self):
        mins = self.Xtr.min(axis=0).values
        maxs = self.Xtr.max(axis=0).values

        steps = torch.linspace(0, 1, steps=8).view(-1,1)

        self.batched_linspace = mins.view(-1) + (maxs-mins).view(1,-1) * steps # shape:8x48000
        
        # to shape:(1,11460,48000)
        X_extended = self.Xtr.view(1, self.Xtr.size(0), -1)
        
        # to shape:(7,1,48000)
        lower_bound = self.batched_linspace[:-1,:].unsqueeze(1)
        upper_bound = self.batched_linspace[1:,:].unsqueeze(1)

        maskes = (lower_bound <= X_extended) & (X_extended <= upper_bound)
        maskes = maskes.view(7,11460,64,750)

        y_extended = self.ytr.view(1,-1,1,1)
        
        # Resulting shape: (7, 64, 750)
        bin_sum = (y_extended * maskes).sum(axis=1)
        
        # Resulting shape: (7, 64, 750)
        bin_counts = maskes.sum(axis=1)

        bin_counts = torch.where(bin_counts==0, torch.ones_like(bin_counts), bin_counts)

        self.fi = bin_sum/bin_counts - self.f0
    
    def run(self, x):
        absolute_mins = self.batched_linspace[0,:].view(64, 750)
        absolute_maxs = self.batched_linspace[-1,:].view(64, 750)

        x = torch.clamp(x, min=absolute_mins, max=absolute_maxs)
        
        # Result shape is: (1, 64, 750)
        x = x.unsqueeze(0)

        # Result shape is: (7, 64, 750)
        lower_bound = self.batched_linspace[:-1,:].view(-1, 64, 750)
        upper_bound = self.batched_linspace[1:,:].view(-1, 64, 750)

        # Result shape is: (7, 64, 750)
        maskes = (lower_bound <= x) & (x <= upper_bound)
        applied_fi = (self.fi * maskes).sum(axis=0)
        out = applied_fi.sum() + self.f0
        return out

        
# hdmr = ANOVAHDMR(Xtr, ytr_)
# hdmr._calculate_fi()            #### took 36 seconds ####

for i in [1,3,4,5,12,701]:        #### took 0 seconds ####
    mimic_yte = hdmr.run(Xte[i])
    mimic_yte, yte_[i], yte[i]
    print(f"test_index:{i:3d} Mimic_out:{mimic_yte:9.4f} Model_out:{yte_[i]:8.4f}, Real_out:{yte[i]}")

test_index:  1 Mimic_out: -63.6558 Model_out: -3.0936, Real_out:1.0
test_index:  3 Mimic_out:   7.6908 Model_out: -1.3469, Real_out:1.0
test_index:  4 Mimic_out:-148.1604 Model_out: -1.9063, Real_out:1.0
test_index:  5 Mimic_out:   6.1208 Model_out:  0.4298, Real_out:1.0
test_index: 12 Mimic_out:   3.3326 Model_out: -0.7022, Real_out:1.0
test_index:701 Mimic_out: 188.1801 Model_out:  0.5151, Real_out:1.0


### Previous Unnececery Attempts 
+ Fitting a RandomForest for mapping a small model. 
+ Employing GAM for a small model.

Since I was working on a small model, logisticRegression with 8 inputs only, the results of HDMR or GAM was close to the real model's results

In [ ]:
# # class HDMR:
# #     def __init__(self, model, X_train, n_bins=8):
# #         self.model = model
# #         self.X = X_train
# #         self.d = X_train.shape[1]
# #         self.n_bins = n_bins
# #         self.f0 = model(X_train).mean() # (11460, 8).mean()
# #         self._compute_first_order()
    
# #     def _compute_first_order(self):
# #         self.fi_funcs = []
# #         for i in range(self.d):
# #             xmin, xmax = self.X[:,i].min(), self.X[:,i].max()
# #             edges = np.linspace(xmin, xmax, self.n_bins+1)
# #             bin_means = np.zeros(self.n_bins)
# #             for b in range(self.n_bins):
# #                 mask = (self.X[:,i] >= edges[b]) & (self.X[:,i] < edges[b+1])
# #                 if mask.sum() > 0:
# #                     bin_means[b] = self.model(self.X[mask]).mean() - self.f0
# #                 else:
# #                     bin_means[b] = 0
# #             self.fi_funcs.append((edges, bin_means))
    
# #     def eval_fi(self, x, i):
# #         edges, means = self.fi_funcs[i]
# #         b = np.digitize(x, edges) - 1
# #         b = np.clip(b, 0, len(means)-1)
# #         return means[b]
    
# #     def decompose(self, X):
# #         N = len(X)
# #         f_i = np.zeros((N, self.d))
# #         for i in range(self.d):
# #             f_i[:, i] = self.eval_fi(X[:, i], i)
# #         f_approx = self.f0 + f_i.sum(axis=1)
# #         return self.f0, f_i, f_approx

# from sklearn.linear_model import LogisticRegression

# clf = LogisticRegression(max_iter=1000)
# clf.fit(Ftr, ytr)

# ypred = clf.predict(Ftest)
# scorr = (ypred == ytest).mean()
# print(f"score: {scorr.item():2f}")

# X_tr = torch.tensor(Ftr, dtype=torch.float32)                               # (11460,8)
# X_te = torch.tensor(Ftest, dtype=torch.float32)
# ptr  = torch.tensor(clf.predict_proba(Ftr)[:, 1], dtype=torch.float32)      # (11460,)
# pte  = torch.tensor(clf.predict_proba(Ftest)[:, 1], dtype=torch.float32)

# from sklearn.ensemble import RandomForestRegressor

# rf = RandomForestRegressor(max_depth = 5)
# rf.fit(X_tr, ptr)
# ypredict = rf.predict(X_te) 
# f"loss {sum((ypredict-pte.numpy())**2)}"



# # import torch
# # import torch.nn as nn

# # class FeatureNet(nn.Module):
# #     def __init__(self, hidden=32):
# #         super().__init__()
# #         self.run = nn.Sequential(
# #             nn.Linear(1, hidden), nn.ReLU(),
# #             nn.Linear(hidden, hidden), nn.ReLU(),
# #             nn.Linear(hidden, 1)
# #         )
# #     def forward(self, x):
# #         return self.run(x)

# # class GAM(nn.Module):
# #     def __init__(self, n_features, hidden=32):
# #         super().__init__()
# #         self.bias = nn.Parameter(torch.zeros(1))
# #         self.feature_nets = nn.ModuleList([FeatureNet(n_features) for i in range(n_features)])
    
# #     def comp1d(self, x):
# #         contributions = torch.cat([net(x[:, i:i+1]) for i, net in enumerate(self.feature_nets)], dim=1)
# #         return contributions 
    
# #     def forward(self, x):
# #         contributions = self.comp1d(x)   
# #         logit = self.bias + contributions.sum(dim=1, keepdim=True) 
# #         out = torch.sigmoid(logit.squeeze(-1)) # (11400,)
# #         return out, contributions

# # gam = GAM(n_features = Ftr.shape[1])
# # opt = torch.optim.Adam(gam.parameters(), lr=1e-2, weight_decay=1e-4) 

# # for epoch in range(200):
# #     gam.train()
# #     opt.zero_grad()
# #     pred, _ = gam(X_tr)
# #     loss = ((pred - ptr)**2).mean()
# #     loss.backward()
# #     opt.step()
# #     if (epoch+1) % 100 == 0:
# #         print(epoch, loss.item())

# # gam.eval()
# # with torch.no_grad():
# #     pred, fi = gam(X_te)
# #     fi = fi - fi.mean(axis=0, keepdim=True)
# #     S = fi.var(axis=0)
# #     S = S/S.sum()
# #     loss = ((pred - pte)**2).mean()
# #     print(f"test loss:{loss.item()}")
# #     print(f"subject importance by order:{S}")


# # # model = lambda x: clf.predict_proba(x)[:, 1]

# # # model(Ftr).shape
# # # model(Ftest)

# # # # 4. Fit HDMR on TRAIN only
# # # hdmr = HDMR(model, Ftr, n_bins=8)

# # # # 5. Decompose a TEST trial
# # # f0, f_i, f_approx = hdmr.decompose(Ftest[0:4])
# # # print(f"Baseline f0: {f0:.3f}")
# # # for i in range(8):
# # #     print(f"  f_{i}: {f_i[0,i]:+.3f}")
# # # print(f"HDMR approx: {f_approx[-1]:.3f}")
# # # print(f"True model:  {model(Ftest[0:1])[-1]:.3f}")

# # # k=[]
# # # for i in range(8):
# # #     e, means = hdmr.fi_funcs[i]
# # #     k.append(means.var())

# # # k = np.asarray(k)
# # # S = k/ k.sum()
# # # S


# # # gam_hard = (pred >= 0.5).float()
# # # clf_hard = (pte >= 0.5).float()
# # # agreement = (gam_hard == clf_hard).float().mean()
# # # print("GAM vs LR agreement:", agreement.item())